# Project Re-evaluation: Statistical Rigor & Foundation Fix

Following a critical review, this notebook re-establishes the project's findings with the following fixes:
1. **Data Leakage Fix:** `MinMaxScaler` fit only on training data.
2. **Sampling Bias Fix:** Random sampling from 100k-row HIGGS buffer.
3. **Statistical Significance:** All benchmarks now use **5 random seeds** (reporting mean ± std).
4. **Honest Baseline:** Classical MLP tuned on the same small dataset (N=1000).

In [ ]:
import sys
sys.path.append('..')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from utils.data_utils import load_higgs, binary_accuracy

SEEDS = [42, 7, 123, 1, 99]
N_SAMPLES = 1000
N_FEATURES = 8

## 1. Rigorous Classical Baseline
We train the Strong MLP across all 5 seeds to establish the real classical performance at N=1000.

In [ ]:
classical_aucs = []

print("Training Classical MLP (5 seeds)...")
for seed in SEEDS:
    X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
        n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, 1)
    )
    
    model = MLPClassifier(
        hidden_layer_sizes=(100, 50, 25), 
        max_iter=500, 
        early_stopping=True, 
        random_state=seed
    )
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    classical_aucs.append(auc)
    print(f"  Seed {seed} | Test AUC: {auc:.4f}")

c_mean, c_std = np.mean(classical_aucs), np.std(classical_aucs)
print(f"\nClassical Result: {c_mean:.4f} ± {c_std:.4f}")

## 2. Rigorous Quantum VQC (Synthesis 2.0)
We re-run the 8-feature, 3-layer Re-uploading circuit across the same 5 seeds.

In [ ]:
dev = qml.device('lightning.qubit', wires=N_FEATURES)

@qml.qnode(dev, interface='autograd')
def circuit(weights, x, n_layers):
    for l in range(n_layers):
        for i in range(len(x)): qml.RY(x[i], wires=i)
        for q in range(len(x)): qml.Rot(*weights[l, q], wires=q)
        for q in range(len(x)):
            qml.CNOT(wires=[q, (q + 1) % len(x)])
    return qml.expval(qml.PauliZ(0))

def train_vqc(X_train, y_train, X_val, y_val, seed):
    pnp.random.seed(seed)
    w = pnp.array(pnp.random.uniform(0, 2*pnp.pi, (3, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    
    opt = qml.AdamOptimizer(stepsize=0.05)
    batch_size = 32
    
    for epoch in range(40):
        # Simple batching
        idx = np.random.permutation(len(X_train))
        for start in range(0, len(X_train), batch_size):
            Xb = X_train[idx][start:start+batch_size]
            yb = y_train[idx][start:start+batch_size].astype(float)
            yb = np.where(yb == 1, 1.0, -1.0)
            
            def cost(w_, b_):
                preds = pnp.array([circuit(w_, x, 3) + b_ for x in Xb])
                return pnp.mean((yb - preds)**2)
            
            w, b = opt.step(cost, w, b)
    return w, b

quantum_aucs = []
print("\nTraining VQC (5 seeds)...")
for seed in SEEDS:
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, np.pi)
    )
    
    w_f, b_f = train_vqc(X_tr, y_tr, X_vl, y_vl, seed)
    
    test_raw = np.array([float(circuit(w_f, x, 3) + b_f) for x in X_te])
    auc = roc_auc_score(y_te, test_raw)
    quantum_aucs.append(auc)
    print(f"  Seed {seed} | Test AUC: {auc:.4f}")

q_mean, q_std = np.mean(quantum_aucs), np.std(quantum_aucs)
print(f"\nQuantum Result: {q_mean:.4f} ± {q_std:.4f}")

## 3. Final Comparison
Is the quantum advantage statistically robust?

In [ ]:
plt.figure(figsize=(8, 5))
labels = ['Classical MLP', 'Quantum VQC']
means = [c_mean, q_mean]
stds = [c_std, q_std]

plt.bar(labels, means, yerr=stds, capsize=10, color=['#B4B2A9', '#7F77DD'], alpha=0.8)
plt.ylabel('Test AUC')
plt.title('Rigorous Benchmarking: Classical vs Quantum (N=1000)')
plt.ylim(0.5, 0.8)
plt.grid(axis='y', alpha=0.3)
plt.show()

advantage = q_mean - c_mean
print(f"\nObserved Mean Advantage: {advantage:.4f}")
if advantage > (c_std + q_std):
    print("STATUS: ADVANTAGE IS STATISTICALLY SIGNIFICANT")
else:
    print("STATUS: ADVANTAGE IS WITHIN NOISE MARGINS")